# RAG Support Chatbot — Milestone 1 (VS Code / local version)
## Data Collection & Preprocessing

**Dataset:** `bitext/Bitext-customer-support-llm-chatbot-training-dataset`

This notebook is a direct port of the original Colab version — same logic,
same PII cleaning, same stratified 80/10/10 split — with Colab-only bits
(`!pip install` magics, widget metadata) removed so it runs cleanly in a
local VS Code + venv setup.

**Before running:** make sure your venv is active and packages are installed —
see the `requirements.txt` / setup cell below.

## Step 0 — One-time setup (run once, then skip)

In [ ]:
# Run this once in your activated venv terminal (NOT as a notebook cell, unless you prefer):
#   pip install datasets pandas numpy scikit-learn requests
#
# Uncomment below if you'd rather install from inside the notebook:
# %pip install datasets pandas numpy scikit-learn requests

In [3]:
import re
import random
import requests
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

DATASET_NAME = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
BASE_URL = "https://datasets-server.huggingface.co/rows"

print('✓ Imports ready')

✓ Imports ready


## Step 1 — Download dataset & clean PII placeholders

In [4]:
def resolve_pii_templates(text: str) -> str:
    """
    Replace Bitext {{placeholder}} tokens with readable, generic text.
    Mirrors the original Colab cleaning logic exactly.
    """
    if not isinstance(text, str):
        return text

    replacements = {
        r"\{\{Order Number\}\}": "your order number",
        r"\{\{Customer Name\}\}": "customer name",
        r"\{\{Customer Support Phone Number\}\}": "our support line",
        r"\{\{Website URL\}\}": "our official website",
        r"\{\{Account ID\}\}": "your account ID",
        r"\{\{Delivery City\}\}": "your delivery location",
    }
    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text)

    # Catch any remaining templates not explicitly listed above
    text = re.sub(r"\{\{.*?\}\}", "information", text)
    return text.strip()


print("Step 1: Downloading full Bitext dataset (approx 5MB)...")
dataset = load_dataset(DATASET_NAME, split="train")
df = pd.DataFrame(dataset)
print(f"  Loaded {len(df):,} rows")

print("Step 2: Applying PII normalization...")
df['instruction_clean'] = df['instruction'].apply(resolve_pii_templates)
df['response_clean']    = df['response'].apply(resolve_pii_templates)

print("Step 3: Filtering 'flags' (isolating real-world noise)...")
# The 'Z' flag in Bitext signifies typos, grammatical errors, and colloquial noise.
# We separate these to test how well the chatbot handles messy/angry typing later.
df_clean = df[~df['flags'].str.contains('Z', na=False)]
df_noisy = df[df['flags'].str.contains('Z', na=False)]

print(f"  -> Clean records (golden data): {len(df_clean):,}")
print(f"  -> Noisy records (typos/errors): {len(df_noisy):,}")

Step 1: Downloading full Bitext dataset (approx 5MB)...


c:\Users\lojyn\OneDrive\Documents\GitHub\NHA-4-231\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lojyn\.cache\huggingface\hub\datasets--bitext--Bitext-customer-support-llm-chatbot-training-dataset. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 26872/2687

  Loaded 26,872 rows
Step 2: Applying PII normalization...
Step 3: Filtering 'flags' (isolating real-world noise)...
  -> Clean records (golden data): 21,586
  -> Noisy records (typos/errors): 5,286


## Step 2 — API-based ingestion utilities (Hugging Face Datasets Server)

In [6]:
def fetch_data_via_api(offset: int = 0, length: int = 100):
    """
    Fetch a specific slice of the dataset via the Hugging Face REST API.
    API maximum length per request is 100.

    Args:
        offset : Row offset to start from
        length : Number of rows to fetch (max 100)

    Returns:
        List of row dicts, or None on failure
    """
    params = {
        "dataset": DATASET_NAME,
        "config": "default",
        "split": "train",
        "offset": offset,
        "length": min(length, 100),
    }
    response = requests.get(BASE_URL, params=params)
    if response.status_code == 200:
        return response.json()['rows']
    else:
        print(f"API Error: Status {response.status_code}")
        return None


def remote_schema_audit() -> bool:
    """
    Audit schema via API to ensure columns match the expected blueprint.
    Raises if the connection fails or required columns are missing.
    """
    print("\n[Task 1] Auditing schema via Hugging Face API...")
    sample_data = fetch_data_via_api(offset=0, length=1)

    if not sample_data:
        raise ConnectionError("Could not connect to Hugging Face API.")

    actual_cols = list(sample_data[0]['row'].keys())
    required_cols = ['flags', 'instruction', 'category', 'intent', 'response']
    missing_cols = [col for col in required_cols if col not in actual_cols]

    if missing_cols:
        raise ValueError(f"Schema Audit Failed! Missing columns: {missing_cols}")

    print(f"✓ Schema Audit Passed. Found expected columns: {actual_cols}")
    return True


def clean_and_format_batch(offset: int = 0, length: int = 3):
    """
    Fetch a batch of data via the API and apply preprocessing rules.

    Args:
        offset : Row offset
        length : Number of rows

    Returns:
        Cleaned DataFrame, or None if fetch failed
    """
    print(f"\n[Task 2] Fetching and cleaning {length} rows...")
    rows_data = fetch_data_via_api(offset, length)

    if not rows_data:
        return None

    data_list = [r['row'] for r in rows_data]
    batch_df = pd.DataFrame(data_list)

    batch_df['instruction_clean'] = batch_df['instruction'].apply(resolve_pii_templates)
    batch_df['response_clean']    = batch_df['response'].apply(resolve_pii_templates)

    return batch_df


def check_intent_distribution_randomized(total_samples: int = 500, max_rows: int = 26872):
    """
    Sample the API at random offsets to get an unbiased intent distribution
    estimate (bypasses the dataset's default sorted ordering).

    Args:
        total_samples : Approx. number of rows to sample in total
        max_rows      : Total rows in the dataset (used to bound random offsets)

    Returns:
        dict mapping intent -> count
    """
    print(f"\n[Task 3] Running randomized intent distribution EDA on {total_samples} samples...")
    intent_counts = {}

    batches = total_samples // 100
    random_offsets = random.sample(range(0, max_rows - 100), batches)
    print(f"  -> Fetching batches at offsets: {random_offsets}")

    for offset in random_offsets:
        rows = fetch_data_via_api(offset=offset, length=100)
        if not rows:
            continue
        for r in rows:
            intent = r['row']['intent']
            intent_counts[intent] = intent_counts.get(intent, 0) + 1

    print("✓ EDA Complete. Estimated Intent Distribution:")
    for intent, count in sorted(intent_counts.items(), key=lambda item: item[1], reverse=True):
        print(f"  - {intent}: {count}")
    return intent_counts

## Step 3 — Run the pipeline checks

In [7]:
print("=== Starting Milestone 1 Pipeline ===")

# 1. Audit the schema
remote_schema_audit()

# 2. Test the cleaning logic on a small batch
processed_df = clean_and_format_batch(offset=0, length=3)
if processed_df is not None:
    print("\n--- Processed Sample Output ---")
    for i, row in processed_df.iterrows():
        print(f"Intent: {row['intent']}")
        print(f"  Raw:   {row['instruction']}")
        print(f"  Clean: {row['instruction_clean']}\n")

# 3. Run the EDA check
check_intent_distribution_randomized(total_samples=500)

print("\n=== Milestone 1 Pipeline Execution Complete ===")

=== Starting Milestone 1 Pipeline ===

[Task 1] Auditing schema via Hugging Face API...
✓ Schema Audit Passed. Found expected columns: ['flags', 'instruction', 'category', 'intent', 'response']

[Task 2] Fetching and cleaning 3 rows...

--- Processed Sample Output ---
Intent: cancel_order
  Raw:   question about cancelling order {{Order Number}}
  Clean: question about cancelling order your order number

Intent: cancel_order
  Raw:   i have a question about cancelling oorder {{Order Number}}
  Clean: i have a question about cancelling oorder your order number

Intent: cancel_order
  Raw:   i need help cancelling puchase {{Order Number}}
  Clean: i need help cancelling puchase your order number


[Task 3] Running randomized intent distribution EDA on 500 samples...
  -> Fetching batches at offsets: [6062, 14464, 22573, 963, 14069]
✓ EDA Complete. Estimated Intent Distribution:
  - edit_account: 200
  - check_refund_policy: 100
  - review: 100
  - change_order: 65
  - cancel_order: 35

=

## Step 4 — Stratified 80/10/10 split (the data going into Milestone 2)

In [8]:
print("Step 4: Executing stratified split on clean data...")

# First, split off 20% for validation + test
train_df, temp_df = train_test_split(
    df_clean,
    test_size=0.20,
    stratify=df_clean['intent'],   # Ensures all 27 intents are equally represented
    random_state=42
)

# Second, split that 20% in half to get 10% validation and 10% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['intent'],
    random_state=42
)

print("\n=== MILESTONE 1 COMPLETE: DATASET VAULTS ===")
print(f"1. Vector Store Index (Train):          {len(train_df):,} rows")
print(f"2. Hyperparameter Tuning (Validation):  {len(val_df):,} rows")
print(f"3. Final Evaluation Vault (Test):       {len(test_df):,} rows")
print(f"4. Robustness Stress-Test (Noisy):       {len(df_noisy):,} rows")

Step 4: Executing stratified split on clean data...

=== MILESTONE 1 COMPLETE: DATASET VAULTS ===
1. Vector Store Index (Train):          17,268 rows
2. Hyperparameter Tuning (Validation):  2,159 rows
3. Final Evaluation Vault (Test):       2,159 rows
4. Robustness Stress-Test (Noisy):       5,286 rows


## Step 5 — Save splits to disk (so Milestone 2 can load them directly)

In [9]:
import os

DATA_DIR = "../data"  # adjust if your notebook lives elsewhere
os.makedirs(DATA_DIR, exist_ok=True)

train_df.to_csv(os.path.join(DATA_DIR, "train_df.csv"), index=False)
val_df.to_csv(os.path.join(DATA_DIR, "val_df.csv"), index=False)
test_df.to_csv(os.path.join(DATA_DIR, "test_df.csv"), index=False)
df_noisy.to_csv(os.path.join(DATA_DIR, "noisy_df.csv"), index=False)

print(f"✓ Saved 4 CSVs to {os.path.abspath(DATA_DIR)}:")
for fname in ["train_df.csv", "val_df.csv", "test_df.csv", "noisy_df.csv"]:
    path = os.path.join(DATA_DIR, fname)
    size_kb = os.path.getsize(path) / 1024
    print(f"  {fname:<16} ({size_kb:.0f} KB)")

✓ Saved 4 CSVs to c:\Users\lojyn\OneDrive\Documents\GitHub\NHA-4-231\data:
  train_df.csv     (23556 KB)
  val_df.csv       (2937 KB)
  test_df.csv      (2963 KB)
  noisy_df.csv     (7064 KB)


## Milestone 1 — Summary

| Deliverable | Status | File |
|---|---|---|
| Processed text corpus | ✓ | `data/train_df.csv`, `val_df.csv`, `test_df.csv` |
| Preprocessing pipeline | ✓ | `resolve_pii_templates()` + flag-based noise filter (this notebook) |
| Support data EDA | ✓ | Randomized intent distribution check (Step 3) |
| Robustness test set | ✓ | `data/noisy_df.csv` (typo/error records) |

---
**Next → Milestone 2 (local version):** Generate embeddings locally with
`sentence-transformers`, build a FAISS vector index, and run retrieval tests —
no Azure account required. See `Milestone_2_Local.ipynb`.